In [ ]:
import json
import re

INPUT_JSON = r"C:\FPTU\doangeo\ocr-pdf-to-text\output\problems\geometry_problems_with_images.json"
OUTPUT_JSON =r"C:\FPTU\doangeo\ocr-pdf-to-text\output\problems\geometry_problems_with_images_tachgiai.json"

CUT_KEYWORDS = [
    "Do ", "Vậy", "Suy ra", "Chứng minh", "Xét ", "Ta có", "Vì "
]

def clean_content(text):
    # tìm (H.x.y)
    match = re.search(r"\(H\.\d+\.\d+\)", text)
    if not match:
        return text.strip()

    h_pos = match.end()
    after = text[h_pos:].strip()

    # nếu sau (H.x.y) còn câu hỏi → giữ
    if any(q in after for q in ["Tính", "Chứng minh", "Hỏi", "Viết", "Tìm"]):
        return text.strip()

    # nếu (H.x.y) nằm giữa câu → giữ
    if not after.startswith("."):
        return text.strip()

    # nếu sau (H.x.y) là lời giải → cắt
    for kw in CUT_KEYWORDS:
        if after.startswith(kw) or f". {kw}" in after:
            return text[:match.start()].strip()

    return text.strip()


with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

for item in data:
    item["content"] = clean_content(item["content"])

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ Đã làm sạch content và lưu vào:", OUTPUT_JSON)
